In [1]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy as sp
import scipy.signal as signal

import math
from sklearn.decomposition import PCA
!pip install elephant
from elephant.gpfa import GPFA

import quantities as pq
from elephant.spike_train_generation import inhomogeneous_poisson_process
import neo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 655.1/655.1 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.4/102.4 kB 7.2 MB/s eta 0:00:00


In [2]:
def generate_spiketrains(instantaneous_rates, num_trials, timestep):
    """
    Parameters
    ----------
    instantaneous_rates : np.ndarray
        Array containing time series.
    timestep :
        Sample period.
    num_steps : int
        Number of timesteps -> max_time = timestep*(num_steps-1).

    Returns
    -------
    spiketrains : list of neo.SpikeTrains
        List containing spiketrains of inhomogeneous Poisson
        processes based on given instantaneous rates.

    """

    spiketrains = []
    for _ in range(num_trials):
        spiketrains_per_trial = []
        for inst_rate in instantaneous_rates:
            anasig_inst_rate = neo.AnalogSignal(inst_rate, sampling_rate=1/timestep, units=pq.Hz)
            spiketrains_per_trial.append(inhomogeneous_poisson_process(anasig_inst_rate))
        spiketrains.append(spiketrains_per_trial)

    return spiketrains

In [3]:
dtype = torch.float
# Check whether GPU is available
if torch.cuda.is_available():
  device = torch.device('cuda')
else:
  device = torch.device('cpu')

In [4]:
dat_dir = '/content/drive/MyDrive/Project/PlasticityDecoding/data/FixedRepresentation'
SPK = []
ptype = []
for rnum in range(0, 30):
    SPK.append(np.load(dat_dir + f'/Spk_an_wn_{rnum}.npy'))
    ptype.append(np.load(dat_dir + f'/Ptype_an_wn_{rnum}.npy'))

SPK = np.concatenate(SPK, axis=0)
ptype = np.concatenate(ptype, axis=0)

# SPK1 = []
# ptype1 = []
# for rnum in range(0,38):
#     SPK1.append(np.load(f'/content/drive/MyDrive/Project/LearningTrajectories_separation/FixedRepresentation/data1/Spk_an_wn_{rnum}.npy'))
#     ptype1.append(np.load(f'/content/drive/MyDrive/Project/LearningTrajectories_separation/FixedRepresentation/data1/Ptype_an_wn_{rnum}.npy'))

# SPK1 = np.concatenate(SPK1, axis=0)
# ptype1 = np.concatenate(ptype1, axis=0)

In [5]:
np.unique(ptype)

array([0., 1., 2., 3., 4.])

In [6]:
num_trials = 60
timestep = 1 * pq.ms

# specify fitting parameters
bin_size = 500 * pq.ms #ms
latent_dimensionality = 2
gpfa_dim = GPFA(bin_size=bin_size, x_dim=latent_dimensionality)

spiketrains_ptype = []
# spiketrains_ptype1 = []
for ii in np.unique(ptype):
    # generate spike trains
    spiketrains_ptype.append(generate_spiketrains(1000 * SPK[np.argwhere(ptype == ii).flatten(),:][:,:], num_trials, timestep))
    # spiketrains_ptype1.append(generate_spiketrains(1000 * SPK1[np.argwhere(ptype1 == ii).flatten()[0:500],:][:,:], num_trials, timestep))
    print(ii)
    break

0.0


In [7]:
# trajectories_ptype = []
# # trajectories_ptype1 = []
# for ii in np.unique(ptype):
#     gpfa_dim.fit(spiketrains_ptype[int(ii)])
#     trajectories_ptype.append(gpfa_dim.transform(spiketrains_ptype[int(ii)]))
#     # trajectories_ptype1.append(gpfa_dim.transform(spiketrains_ptype1[int(ii)]))

In [8]:
len(spiketrains_ptype[0])

60

In [9]:
len(spiketrains_ptype[0][0])

616

In [10]:
spiketrains_ptype[0][0]

[SpikeTrain containing 80 spikes; units s; datatype float64 
 time: 0.0 s to 15.0 s,
 SpikeTrain containing 73 spikes; units s; datatype float64 
 time: 0.0 s to 15.0 s,
 SpikeTrain containing 54 spikes; units s; datatype float64 
 time: 0.0 s to 15.0 s,
 SpikeTrain containing 84 spikes; units s; datatype float64 
 time: 0.0 s to 15.0 s,
 SpikeTrain containing 91 spikes; units s; datatype float64 
 time: 0.0 s to 15.0 s,
 SpikeTrain containing 63 spikes; units s; datatype float64 
 time: 0.0 s to 15.0 s,
 SpikeTrain containing 95 spikes; units s; datatype float64 
 time: 0.0 s to 15.0 s,
 SpikeTrain containing 72 spikes; units s; datatype float64 
 time: 0.0 s to 15.0 s,
 SpikeTrain containing 93 spikes; units s; datatype float64 
 time: 0.0 s to 15.0 s,
 SpikeTrain containing 76 spikes; units s; datatype float64 
 time: 0.0 s to 15.0 s,
 SpikeTrain containing 94 spikes; units s; datatype float64 
 time: 0.0 s to 15.0 s,
 SpikeTrain containing 72 spikes; units s; datatype float64 
 tim